# Set Up

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("temp/spotify-million")

print("Downloaded here: ", path)

/Users/rushikesh/python_files/vscode/entity2Vector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 5.20G/5.20G [10:40<00:00, 8.72MB/s]  

Extracting files...


Downloaded here:  /Users/rushikesh/.cache/kagglehub/datasets/himanshuwagh/spotify-million/versions/1


In [64]:
import os
import json
from glob import glob
import tqdm

data_dir = path + "/data/" # returned by kagglehub
playlist_dict = {}

json_files = glob(os.path.join(data_dir, "*.json"))

for jf in tqdm.tqdm(json_files):
    with open(jf, "r") as f:
        data = json.load(f)
        for playlist in data["playlists"]:
            pid = playlist["pid"]
            tracks = [t["track_uri"] for t in playlist["tracks"]]
            playlist_dict[pid] = tracks

print(f"Total playlists: {len(playlist_dict)}")

100%|██████████| 1000/1000 [06:28<00:00,  2.57it/s] 

Total playlists: 1000000


In [ ]:
rm -rf "/.cache/kagglehub/datasets"

In [6]:
import pickle
import os

local_path = '/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/'

save_path = os.path.join(local_path, "playlist_dict.pkl")

with open(save_path, "wb") as f:
    pickle.dump(playlist_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved to:", save_path)

Saved to: /Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/playlist_dict.pkl


# Data Processing

In [1]:
%cd "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/"

/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python


In [57]:
import pickle

load_path = "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/playlist_dict.pkl"

with open(load_path, "rb") as f:
    playlist_dict = pickle.load(f)

print("Loaded playlists:", len(playlist_dict))

Loaded playlists: 1000000


In [ ]:
import os
import json
from glob import glob
import tqdm

data_dir = path + "/data/"

# only keep tracks present in tokenizer vocab
valid_uris = set(tokenizer.stoi.keys())

track_lookup = {}

json_files = glob(os.path.join(data_dir, "*.json"))

for jf in tqdm.tqdm(json_files):
    with open(jf, "r") as f:
        data = json.load(f)

        for playlist in data["playlists"]:
            for t in playlist["tracks"]:
                uri = t["track_uri"]

                if uri in valid_uris and uri not in track_lookup:
                    track_lookup[uri] = t["track_name"]

100%|██████████| 1000/1000 [02:08<00:00,  7.80it/s]


In [ ]:
import pickle
import os

local_path = '/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/'

save_path = os.path.join(local_path, "track_lookup.pkl")

with open(save_path, "wb") as f:
    pickle.dump(track_lookup, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved to:", save_path)

Saved to: /Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/track_lookup.pkl


# Tokenizer

In [4]:
import torch
from utils.Tokenizer.Tokenizer import simpleTokenizer

In [5]:
tokenizer = simpleTokenizer(playlist_dict)

In [8]:
len(tokenizer.itos)

2262292

# stats 

In [14]:
freq = {}

for ls in playlist_dict.values():

    for _ in ls:

        freq[tokenizer.stoi[_]] = freq.get(tokenizer.stoi[_], 0) + 1
    


In [22]:
import numpy as np

vals = np.array(list(freq.values()))

print("tokens       :", len(vals))
print("total_count  :", vals.sum())
print("mean         :", vals.mean())
print("std          :", vals.std())
print("min          :", vals.min())
print("25%          :", np.percentile(vals, 25))
print("median       :", np.median(vals))
print("75%          :", np.percentile(vals, 75))
print("max          :", vals.max())

tokens       : 2262292
total_count  : 66346428
mean         : 29.32708421370893
std          : 360.6910593436592
min          : 1
25%          : 1.0
median       : 2.0
75%          : 5.0
max          : 46574


In [ ]:
## Not all the songs are that frequnt
## Infact most of them are very rare

## lets drop them from the data

## update the playlist dict after dropping tokens whose frequency is less than 50

In [28]:
min_freq = 100
keep = {tok for tok, c in freq.items() if c >= min_freq}

print("kept tokens:", len(keep))

kept tokens: 70229


In [29]:
playlist_dict_new = {}
for pid, tracks in playlist_dict.items():
    playlist_dict_new[pid] = [t for t in tracks if tokenizer.stoi[t] in keep]

In [30]:
import pickle
import os

local_path = '/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/'

save_path = os.path.join(local_path, "playlist_dict_prunned.pkl")

with open(save_path, "wb") as f:
    pickle.dump(playlist_dict_new, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved to:", save_path)

Saved to: /Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/playlist_dict_prunned.pkl


# Prunned Data

In [1]:
%cd "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/"

/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python


In [2]:
import pickle

load_path = "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/playlist_dict_prunned.pkl"

with open(load_path, "rb") as f:
    playlist_dict_new = pickle.load(f)

print("Loaded playlists:", len(playlist_dict_new))

Loaded playlists: 1000000


In [3]:
import torch
from utils.Tokenizer.Tokenizer import simpleTokenizer

In [4]:
tokenizer = simpleTokenizer(playlist_dict_new)

In [5]:
len(tokenizer.itos)

70229

In [6]:
freq = {}

for ls in playlist_dict_new.values():

    for _ in ls:

        freq[tokenizer.stoi[_]] = freq.get(tokenizer.stoi[_], 0) + 1
    


In [7]:
import numpy as np

vals = np.array(list(freq.values()))

print("tokens       :", len(vals))
print("total_count  :", vals.sum())
print("mean         :", vals.mean())
print("std          :", vals.std())
print("min          :", vals.min())
print("25%          :", np.percentile(vals, 25))
print("median       :", np.median(vals))
print("75%          :", np.percentile(vals, 75))
print("max          :", vals.max())

tokens       : 70229
total_count  : 53521564
mean         : 762.1006137065884
std          : 1905.8007094890265
min          : 100
25%          : 144.0
median       : 240.0
75%          : 547.0
max          : 46574


In [9]:
from collections import defaultdict
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix, vstack
import tqdm


def build_cooccurrence(
    playlists,
    vocab_size,
    window=5,
    chunk_size=5000,   # number of playlists per chunk
    assume_ids=True    # set False if playlists contain strings
):
    """
    Symmetric co-occurrence within a context window.

    playlists: dict[pid] -> list[int] or list[str]
    vocab_size: vocabulary size
    window: context window on each side
    chunk_size: process playlists in chunks (reduces CPU/memory spikes)
    assume_ids: if True, tracks are already token ids (FASTER)
    """

    playlist_values = list(playlists.values())
    num_playlists = len(playlist_values)

    chunks = []

    for start in tqdm.tqdm(range(0, num_playlists, chunk_size)):
        end = min(start + chunk_size, num_playlists)
        chunk_lists = playlist_values[start:end]

        cooc_counts = defaultdict(int)

        for tracks in chunk_lists:
            n = len(tracks)

            for i in range(n):
                src = tokenizer.stoi[tracks[i]]

                left = max(0, i - window)
                right = min(n, i + window + 1)

                for j in range(left, right):
                    if i == j:
                        continue

                    tgt = tokenizer.stoi[tracks[j]]

                    cooc_counts[(src, tgt)] += 1

        # --- convert chunk to sparse ---
        if cooc_counts:
            rows, cols, data = zip(
                *((i, j, c) for (i, j), c in cooc_counts.items())
            )
            chunk_mat = coo_matrix(
                (data, (rows, cols)),
                shape=(vocab_size, vocab_size),
            ).tocsr()
            chunks.append(chunk_mat)

    # --- merge all chunks ---
    if len(chunks) == 1:
        return chunks[0]

    print("Merging chunks...")
    return sum(chunks, csr_matrix((vocab_size, vocab_size)))

In [10]:
## Example
t =build_cooccurrence(
    playlists={'test':['spotify:track:6QHYEZlm9wyfXfEM1vSu1P','spotify:track:3RkQ3UwOyPqpIiIvGVewuU','spotify:track:3RkQ3UwOyPqpIiIvGVewuU','spotify:track:3RkQ3UwOyPqpIiIvGVewuU'], 
               'test2': ['spotify:track:3RkQ3UwOyPqpIiIvGVewuU','spotify:track:0ju1jP0cSPJ8tmojYBEI89','spotify:track:0ju1jP0cSPJ8tmojYBEI89','spotify:track:6QHYEZlm9wyfXfEM1vSu1P']},
    vocab_size=len(tokenizer.stoi),
    window=5
)

t[0,1]

100%|██████████| 1/1 [00:00<00:00, 173.81it/s]


np.int64(4)

In [14]:
import random
import pickle

pids = sorted(playlist_dict_new.keys())
random.seed(42)
random.shuffle(pids)

split = int(0.95 * len(pids))
test_ids = pids[split:]

with open("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/test_pids.pkl", "wb") as f:
    pickle.dump(test_ids, f)

In [16]:
import pickle

with open("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/test_pids.pkl", "rb") as f:
    test_ids = set(pickle.load(f))

train_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid not in test_ids
}

test_playlists = {
    pid: tracks for pid, tracks in playlist_dict_new.items()
    if pid in test_ids
}

In [18]:
print(f"Number of documents in Train: {len(train_playlists)}")
print(f"Number of documents in Test : {len(test_playlists)}")

Number of documents in Train: 950000
Number of documents in Test : 50000


In [19]:
cooc = build_cooccurrence(
    playlists=train_playlists,
    vocab_size=len(tokenizer.stoi),
    window=5
)

print(cooc.shape, cooc.nnz)

100%|██████████| 190/190 [06:08<00:00,  1.94s/it]


Merging chunks...
(70229, 70229) 144431313


In [20]:
from scipy.sparse import save_npz

save_npz("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/cooc_matrix.npz", cooc)

# COOC Matrix

In [27]:
from scipy.sparse import load_npz

cooc = load_npz("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/cooc_matrix.npz")

In [28]:
cooc[0,1001], cooc[1001,0]

(np.float64(2.0), np.float64(2.0))

In [29]:
print(f"Filled : {100*cooc.nnz/(cooc.shape[0] * cooc.shape[1]):.2f}%")

Filled : 2.93%


# Postitive pointwise mutual information

In [30]:
import numpy as np
from scipy.sparse import csr_matrix

def csr_to_ppmi(cooc: csr_matrix):
    cooc = cooc.tocsr().astype(np.float64)

    # totals
    total = cooc.sum()
    row_sum = np.array(cooc.sum(axis=1)).flatten()
    col_sum = np.array(cooc.sum(axis=0)).flatten()

    # avoid divide-by-zero
    row_sum[row_sum == 0] = 1
    col_sum[col_sum == 0] = 1

    # iterate over nonzeros only
    rows, cols = cooc.nonzero()
    data = cooc.data

    # PMI
    pmi = np.log((data * total) / (row_sum[rows] * col_sum[cols]))

    # PPMI
    pmi[pmi < 0] = 0

    return csr_matrix((pmi, (rows, cols)), shape=cooc.shape)

In [31]:
ppmi = csr_to_ppmi(cooc)

# SVD

In [36]:
from sklearn.decomposition import TruncatedSVD

k = 128  # embedding dim

svd = TruncatedSVD(n_components=k, random_state=42)

# 1. Fit SVD

U_sigma = svd.fit_transform(ppmi)  # This is U * Sigma
sigma = svd.singular_values_

# 2. Compute U * sqrt(Sigma)

Embeddings = U_sigma / np.sqrt(sigma)

In [46]:
Embeddings_norm = Embeddings/ np.linalg.norm(Embeddings, axis=1, keepdims=True)

In [59]:
np.save("/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/embeddings_norm.npy", Embeddings_norm)

In [ ]:
# 3. Optional: Get the context vectors (V * sqrt(Sigma))
# V = svd.components_.T 
# C = V * np.sqrt(sigma)

# Evaluation

In [44]:
import random

def build_eval_dict(test_playlists, window=5, n_pos=2, n_neg=3, seed=42):
    rng = random.Random(seed)

    # pool for negatives (all tokens seen in test)
    all_tokens = list({tokenizer.stoi[t] for tracks in test_playlists.values() for t in tracks})

    eval_data = {}

    for pid, tracks in test_playlists.items():
        if len(tracks) < window + 2:
            continue

        # pick pivot safely
        i = rng.randint(0, len(tracks) - 1)
        pivot = tokenizer.stoi[tracks[i]]

        # context window (both sides)
        left = max(0, i - window)
        right = min(len(tracks), i + window + 1)
        context = [tokenizer.stoi[t] for idx, t in enumerate(tracks[left:right]) if idx + left != i]

        if len(context) < n_pos:
            continue

        positives = rng.sample(context, n_pos)

        # negatives: not in this playlist context
        neg_pool = [t for t in all_tokens if t not in context and t != pivot]
        negatives = rng.sample(neg_pool, n_neg)

        eval_data[pid] = {
            "pivot": pivot,
            "positives": positives,
            "negatives": negatives,
        }

    return eval_data

In [45]:
eval_dict = build_eval_dict(test_playlists, window=5)
print(len(eval_dict))

46929


In [48]:
import numpy as np

def evaluate_embeddings(X, eval_dict, k_recall=2):
    recalls = []
    ranks = []

    for ex in tqdm.tqdm(eval_dict.values()):
        pivot = ex["pivot"]
        positives = ex["positives"]
        negatives = ex["negatives"]

        candidates = positives + negatives

        pivot_vec = X[pivot]
        sims = [pivot_vec @ X[t] for t in candidates]

        # rank (descending similarity)
        order = np.argsort(sims)[::-1]

        # positions of positives
        pos_ranks = []
        for idx in range(len(positives)):
            rank = np.where(order == idx)[0][0] + 1  # 1-based
            pos_ranks.append(rank)

        # Recall@k
        recall = sum(r <= k_recall for r in pos_ranks) / len(pos_ranks)
        recalls.append(recall)

        # Average rank
        ranks.extend(pos_ranks)

    return {
        "recall@{}".format(k_recall): np.mean(recalls),
        "avg_rank": np.mean(ranks),
    }

In [50]:
metrics = evaluate_embeddings(Embeddings_norm, eval_dict, k_recall=2)
print(metrics)

100%|██████████| 46929/46929 [00:00<00:00, 72060.45it/s]

{'recall@2': np.float64(0.8746510686355985), 'avg_rank': np.float64(1.731104434358286)}


# Example

In [72]:
import pickle

load_path = "/Users/rushikesh/python_files/vscode/entity2Vector/src/main/python/data/track_lookup.pkl"

with open(load_path, "rb") as f:
    track_lookup = pickle.load(f)

print("Loaded playlists:", len(track_lookup))

Loaded playlists: 70229


In [76]:
import numpy as np

def most_similar(token_id, embeddings, top_k=10):
    """
    Returns top_k most similar songs for a given token_id
    """
    target_vec = embeddings[token_id]

    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(target_vec)
    sims = embeddings @ target_vec / norms  # cosine similarity

    sims[token_id] = -1  # exclude target song itself
    top_indices = np.argsort(-sims)[:top_k]

    similar_songs = [track_lookup[tokenizer.itos[i]] for i in top_indices]

    return similar_songs


In [98]:
# Song query
query = "They don't care about us"

# Find all URIs whose track name contains the query (case-insensitive)
uris = [u for u, name in track_lookup.items() if query.lower() in name.lower()]

if not uris:
    print("No matching tracks found.")
else:
    for u in uris:
        token_id = tokenizer.stoi.get(u)
        if token_id is not None:
            print(f"Found token_id {token_id} for track '{track_lookup[u]}' (URI: {u})")
        else:
            print(f"Track '{track_lookup[u]}' not in tokenizer vocabulary")

Found token_id 21432 for track 'They Don't Care About Us' (URI: spotify:track:3wuCCNCnBhJlwkIJTBZFiv)


In [99]:

# Example usage:
# Suppose test_token_ids is a list of token IDs from your test playlists
test_token_ids = [21432]  # replace with actual token IDs

for tid in test_token_ids:
    print(f"Most similar songs to token {tid}:")
    print(most_similar(tid, Embeddings_norm, top_k=10))
    print()

Most similar songs to token 21432:
['Who Is It', 'Leave Me Alone - 2012 Remaster', 'Dangerous', 'Speed Demon - 2012 Remaster', 'Earth Song', 'Jam', 'Bad - 2012 Remaster', 'Give In to Me', 'Black or White', 'The Way You Make Me Feel - 2012 Remaster']



In [ ]:
example = eval_dict[549018]

In [ ]:
tokenizer.itos[example['pivot']]

'spotify:track:5BOZ4skcMubA0R6RD4zf64'